In [3]:
import os
from pathlib import Path

DATA_ROOT = Path("Data")
from read.iop import IOPParser  # ✅ 确保 read/iop.py 里定义了 IOPParser


def parse_iop_article(filepath, output_folder):
    """解析单篇 IOP HTML 文件并将 paragraph 写入 txt"""
    try:
        # 1. 初始化解析器
        iop_parser = IOPParser(filepath)

        # 2. 解析元信息（依赖我们在 IOPParser.parse_meta 里返回 dict）
        meta_data = iop_parser.parse_meta()
        print(f"📄 Title: {meta_data.get('title', 'N/A')}")
        print(f"📘 Journal: {meta_data.get('journal', 'N/A')}")
        # print(f"🧾 Abstract: {meta_data.get('abstract', 'N/A')}")
        # print(f"📅 Date: {meta_data.get('date', 'N/A')}")

        # 3. 解析段落（IOPParser.parse_paragraphs 返回的是元素列表）
        para_elements = iop_parser.parse_paragraphs()
        paragraph_texts = []

        for el in para_elements:
            # lxml 元素：有 text_content 方法
            if hasattr(el, "text_content"):
                txt = el.text_content().strip()
            else:
                txt = str(el).strip()

            if txt:
                paragraph_texts.append(txt)

        print(f"📝 段落数: {len(paragraph_texts)}")

        # 4. 构建输出文件名
        base_name = os.path.basename(filepath)
        # 去掉 .html / .htm / .xhtml 等后缀
        for ext in [".html", ".htm", ".xhtml"]:
            if base_name.lower().endswith(ext):
                base_name = base_name[: -len(ext)]
                break

        # 处理成安全文件名
        safe_name = "".join(c for c in base_name if c.isalnum() or c in (" ", "_", "-"))
        output_path = os.path.join(output_folder, f"{safe_name}.txt")

        # 5. 写入到 txt 文件
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(f"Title: {meta_data.get('title', '')}\n")
            f.write(f"Journal: {meta_data.get('journal', '')}\n")
            f.write(f"Date: {meta_data.get('date', '')}\n")
            f.write(f"Abstract: {meta_data.get('abstract', '')}\n\n")
            f.write("Paragraphs:\n")
            for para in paragraph_texts:
                f.write(para + "\n\n")

        print(f"✅ 已保存段落 → {output_path}\n")

    except Exception as e:
        print(f"❌ 解析失败：{filepath}\n错误信息：{e}\n")


def batch_parse_iop_folder(input_folder, output_folder):
    """批量解析整个文件夹下的 IOP HTML 文件（带重复检查功能）"""
    os.makedirs(output_folder, exist_ok=True)

    # 1. 获取所有待处理文件
    html_files = [
        f for f in os.listdir(input_folder)
        if f.lower().endswith((".html", ".htm", ".xhtml"))
    ]

    if not html_files:
        print("⚠️ 未找到 HTML 文件，请检查路径。")
        return

    print(f"🚀 开始批量解析 IOP 文献，共 {len(html_files)} 篇...\n")

    for html_file in html_files:
        # --- 新增：自动检查逻辑 ---
        # 模拟 parse_iop_article 内部的文件名生成逻辑
        base_name = html_file
        for ext in [".html", ".htm", ".xhtml"]:
            if base_name.lower().endswith(ext):
                base_name = base_name[: -len(ext)]
                break
        
        safe_name = "".join(c for c in base_name if c.isalnum() or c in (" ", "_", "-"))
        output_path = os.path.join(output_folder, f"{safe_name}.txt")

        # 判断文件是否存在且大小不为 0（代表已经成功处理过）
        if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
            # print(f"⏭️  已跳过: {html_file}")
            continue
        # ------------------------

        file_path = os.path.join(input_folder, html_file)
        parse_iop_article(file_path, output_folder)

    print("🎯 全部解析完成！")


if __name__ == "__main__":
    # 👉 换成你的 IOP HTML 文件夹路径
    input_folder = DATA_ROOT / "IOP" / "source"   # 📂 输入 HTML 文件夹路径
    output_folder = DATA_ROOT / "IOP" / "txt"     # 📂 输出 TXT 文件夹路径

    batch_parse_iop_folder(input_folder, output_folder)


🚀 开始批量解析 IOP 文献，共 336 篇...

🎯 全部解析完成！
